# Minimum-width minimizer for one AR(3) process at $k=4$

This notebook generates one stationary AR(3) process with innovations $\varepsilon_t\sim\operatorname{Unif}[-1,1]$, constructs only the $k=p+1=4$ sliding-window point cloud, and minimizes its empirical directional width on the unit sphere.

The optimization uses 12 reproducible random starts for projected subgradient descent and retains the run with the smallest width. Because width is invariant under $u\mapsto-u$, the reported sign is aligned with the population AR direction $(1,-\phi_1,-\phi_2,-\phi_3)$.

In [ ]:
import sys                                                                                          ### INFRASTRUCTURE
import time
from pathlib import Path                                                                            ### INFRASTRUCTURE

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.stats import mannwhitneyu, spearmanr

AR_CODE_DIR = next(                                                                                 ### INFRASTRUCTURE
    (                                                                                               ### INFRASTRUCTURE
        candidate.resolve()                                                                         ### INFRASTRUCTURE
        for root in (Path.cwd(), *Path.cwd().parents)                                               ### INFRASTRUCTURE
        for candidate in (root / "Code" / "Code_AR_processes", root)                                ### INFRASTRUCTURE
        if (candidate / "ar_tools/functions.py").is_file()                                          ### INFRASTRUCTURE
    ),                                                                                              ### INFRASTRUCTURE
    None,                                                                                           ### INFRASTRUCTURE
)                                                                                                   ### INFRASTRUCTURE
if AR_CODE_DIR is None:                                                                             ### INFRASTRUCTURE
    raise FileNotFoundError(                                                                        ### INFRASTRUCTURE
        "Could not locate Code/Code_AR_processes from the current directory."                       ### INFRASTRUCTURE
    )                                                                                               ### INFRASTRUCTURE
PROJECT_ROOT = AR_CODE_DIR.parents[1]                                                               ### INFRASTRUCTURE
if str(PROJECT_ROOT) not in sys.path:                                                               ### INFRASTRUCTURE
    sys.path.insert(0, str(PROJECT_ROOT))                                                           ### INFRASTRUCTURE

from Code.Code_AR_processes.ar_tools.descent import (
    batched_geometric_width_curves,
    projected_subgradient_empirical_width,
)
from Code.Code_AR_processes.ar_tools.functions import (
    jones_ar_coefficients,
    simulate_uniform_ar,
    sliding_window_point_cloud,
)

np.set_printoptions(precision=8, suppress=True)                                                     ### REPORTING

from Code.Code_AR_processes.ar_tools.coefficient_errors import coefficient_mae


In [ ]:
order = 3
embedding_dimension = order + 1
sample_size = 2_000
burnin = 1_000
coefficient_seed = 20260910
process_seed = 20260911

phi = jones_ar_coefficients(order, rng=coefficient_seed)
ar3_series = simulate_uniform_ar(
    phi,
    sample_size=sample_size,
    burnin=burnin,
    half_width=1.0,
    rng=process_seed,
)
point_cloud_k4 = sliding_window_point_cloud(
    ar3_series,
    embedding_dimension=embedding_dimension,
)

print(f"AR coefficients: {phi}")                                                                    ### REPORTING
print(f"Embedding dimension: k = {embedding_dimension}")                                            ### REPORTING
print(f"Point-cloud shape: {point_cloud_k4.shape}")                                                 ### REPORTING

In [ ]:
number_of_starts = 12
optimization_iterations = 5_000
optimizer_seed = 20260912
initial_step = 0.2

# Use reproducible normalized random directions for every start.
initial_direction_generator = np.random.default_rng(optimizer_seed)
initial_directions = initial_direction_generator.normal(
    size=(number_of_starts, embedding_dimension)
)
initial_directions /= np.linalg.norm(initial_directions, axis=1, keepdims=True)

descent_results = [
    projected_subgradient_empirical_width(
        point_cloud_k4,
        iterations=optimization_iterations,
        initial_step=initial_step,
        patience=1_200,
        rng=optimizer_seed + start_index,
        initial_direction=initial_direction,
        step_schedule="geometric",
        geometric_decay=0.997,
    )
    for start_index, initial_direction in enumerate(initial_directions)
]

start_minimum_widths = np.array([
    result["minimum_width"] for result in descent_results
])
best_start = int(np.argmin(start_minimum_widths))
best_result = descent_results[best_start]
estimated_minimizer = best_result["direction"].copy()

# Fix the otherwise arbitrary sign of the minimizing direction.
population_ar_direction = np.concatenate(([1.0], -phi))
population_ar_direction /= np.linalg.norm(population_ar_direction)
if estimated_minimizer @ population_ar_direction < 0.0:
    estimated_minimizer *= -1.0

print(f"Absolute initial step eta_0: {initial_step:.10f}")                                       ### REPORTING
print(f"Best start: {best_start + 1} of {number_of_starts}")                                        ### REPORTING
print(f"Minimum empirical width: {best_result['minimum_width']:.10f}")                              ### REPORTING
print("Minimizing unit direction u_hat:")                                                           ### REPORTING
print(estimated_minimizer)                                                                          ### REPORTING
print(f"Norm of u_hat: {np.linalg.norm(estimated_minimizer):.10f}")                                 ### REPORTING
print("Population AR direction (diagnostic):")                                                      ### REPORTING
print(population_ar_direction)                                                                      ### REPORTING
print(                                                                                              ### REPORTING
    "Absolute cosine similarity: "                                                                  ### REPORTING
    f"{abs(estimated_minimizer @ population_ar_direction):.10f}"                                    ### REPORTING
)                                                                                                   ### REPORTING

## Monte Carlo coefficient-recovery experiment

For each of 50 replications, this experiment independently draws stationary AR(3) coefficients with Jones' method, simulates a length-$2{,}000$ process with $\operatorname{Unif}[-1,1]$ innovations, and estimates the minimum-width direction using only the $k=4$ embedding. The geometric schedule uses absolute steps $\eta_t=0.2(0.997)^t$.

The coefficient estimate is

$$\widehat\phi_j=-\frac{\widehat u_j}{\widehat u_0},\qquad j=1,2,3.$$

As a classical benchmark, the same point cloud is fitted by $L^\infty$ (Chebyshev) regression. It solves the linear program

$$\min_{\phi,c,r} r \quad\text{subject to}\quad \left|X_t-\sum_{j=1}^3\phi_jX_{t-j}-c\right|\le r \quad\text{for every }t.$$

This minimizes the unnormalized residual range, whereas the proposed method minimizes directional width over the sphere. The LP is therefore used only as a coefficient-estimation benchmark.

Both methods receive exactly the same coefficient draws and simulated paths. The per-fit score is the mean absolute error across the three coefficients; the overall MAE averages that score across all replications.

In [ ]:
from Code.Code_AR_processes.ar_tools.lp_widths import chebyshev_ar_fit


replication_count = 50
experiment_seed = 20260913
experiment_rows = []

for replication in range(replication_count):
    process_generator = np.random.default_rng(
        np.random.SeedSequence([experiment_seed, replication, 0])
    )
    true_phi = jones_ar_coefficients(order, rng=process_generator)
    replicated_series = simulate_uniform_ar(
        true_phi,
        sample_size=sample_size,
        burnin=burnin,
        half_width=1.0,
        rng=process_generator,
    )
    replicated_point_cloud = sliding_window_point_cloud(
        replicated_series, embedding_dimension=embedding_dimension
    )

    geometric_start_time = time.perf_counter()
    replicated_initial_step = initial_step
    _, replicated_widths, _, replicated_directions = (
        batched_geometric_width_curves(
            replicated_series,
            embedding_dimension,
            seed=experiment_seed,
            replication=replication,
            starts=number_of_starts,
            iterations=optimization_iterations,
            initial_step=replicated_initial_step,
            geometric_decay=0.997,
            return_directions=True,
        )
    )
    replicated_minimizer = replicated_directions[embedding_dimension - 1]
    if abs(replicated_minimizer[0]) <= np.finfo(float).eps:
        raise RuntimeError(
            f"Cannot recover coefficients in replication {replication + 1}: "
            "the first minimizer component is numerically zero."
        )
    estimated_phi = (
        -replicated_minimizer[1:] / replicated_minimizer[0]
    )
    true_direction = np.concatenate(([1.0], -true_phi))
    true_direction /= np.linalg.norm(true_direction)
    direction_cosine_similarity = abs(replicated_minimizer @ true_direction)
    geometric_runtime = time.perf_counter() - geometric_start_time
    absolute_errors = np.abs(estimated_phi - true_phi)

    chebyshev_start_time = time.perf_counter()
    chebyshev_phi, chebyshev_center, chebyshev_radius = (
        chebyshev_ar_fit(replicated_point_cloud)
    )
    chebyshev_runtime = time.perf_counter() - chebyshev_start_time
    chebyshev_absolute_errors = np.abs(chebyshev_phi - true_phi)

    row = {
        "replication": replication + 1,
        "minimum_width": replicated_widths[embedding_dimension - 1],
        "coefficient_mae": coefficient_mae(estimated_phi, true_phi),
        "initial_step": replicated_initial_step,
        "abs_u_0": abs(replicated_minimizer[0]),
        "geometric_runtime_seconds": geometric_runtime,
        "direction_cosine_similarity": direction_cosine_similarity,
        "chebyshev_coefficient_mae": coefficient_mae(chebyshev_phi, true_phi),
        "chebyshev_residual_center": chebyshev_center,
        "chebyshev_residual_radius": chebyshev_radius,
        "chebyshev_runtime_seconds": chebyshev_runtime,
    }
    for coefficient_index in range(order):
        suffix = coefficient_index + 1
        row[f"true_phi_{suffix}"] = true_phi[coefficient_index]
        row[f"estimated_phi_{suffix}"] = estimated_phi[coefficient_index]
        row[f"absolute_error_phi_{suffix}"] = absolute_errors[coefficient_index]
        row[f"chebyshev_phi_{suffix}"] = chebyshev_phi[coefficient_index]
        row[f"chebyshev_absolute_error_phi_{suffix}"] = (
            chebyshev_absolute_errors[coefficient_index]
        )
    experiment_rows.append(row)

    if (replication + 1) % 10 == 0:                                                                 ### REPORTING
        print(f"Completed {replication + 1}/{replication_count} processes")                         ### REPORTING

ar3_mae_results = pd.DataFrame(experiment_rows)

In [ ]:
absolute_error_columns = [
    f"absolute_error_phi_{index}" for index in range(1, order + 1)
]
chebyshev_absolute_error_columns = [
    f"chebyshev_absolute_error_phi_{index}"
    for index in range(1, order + 1)
]
method_columns = {
    "Minimum width": {
        "process_mae": "coefficient_mae",
        "absolute_errors": absolute_error_columns,
        "runtime": "geometric_runtime_seconds",
    },
    "LP Chebyshev": {
        "process_mae": "chebyshev_coefficient_mae",
        "absolute_errors": chebyshev_absolute_error_columns,
        "runtime": "chebyshev_runtime_seconds",
    },
}
comparison_rows = []
for method, columns in method_columns.items():
    process_mae = ar3_mae_results[columns["process_mae"]]
    comparison_rows.append({
        "method": method,
        "overall_coefficient_mae": float(
            ar3_mae_results[columns["absolute_errors"]].to_numpy().mean()
        ),
        "median_process_mae": float(process_mae.median()),
        "maximum_process_mae": float(process_mae.max()),
        "failures_mae_above_1": int((process_mae > 1.0).sum()),
        "mean_runtime_seconds": float(
            ar3_mae_results[columns["runtime"]].mean()
        ),
    })
comparison_summary = pd.DataFrame(comparison_rows).set_index("method")

mae_by_coefficient = pd.DataFrame(
    {
        method: [
            ar3_mae_results[column].mean()
            for column in columns["absolute_errors"]
        ]
        for method, columns in method_columns.items()
    },
    index=[f"phi_{index}" for index in range(1, order + 1)],
)
lp_win_count = int(
    (
        ar3_mae_results["chebyshev_coefficient_mae"]
        < ar3_mae_results["coefficient_mae"]
    ).sum()
)

print("Paired comparison on the same 50 processes:")                                                ### REPORTING
print(comparison_summary.to_string())                                                               ### REPORTING
print(f"\nLP Chebyshev has lower process MAE in {lp_win_count}/50 runs.")                           ### REPORTING
print(                                                                                              ### REPORTING
    "eta_0 range: "                                                                                 ### REPORTING
    f"{ar3_mae_results['initial_step'].min():.8f} to "                                              ### REPORTING
    f"{ar3_mae_results['initial_step'].max():.8f}"                                                  ### REPORTING
)                                                                                                   ### REPORTING
print("MAE by coefficient and method:")                                                             ### REPORTING
print(mae_by_coefficient.to_string())                                                               ### REPORTING

ar3_mae_results                                                                                     ### REPORTING

## When does minimum width beat LP Chebyshev?

This exploratory paired analysis defines a win by lower coefficient MAE on the same simulated path. Its additional regular-regime comparison uses fits with MAE at most one; all fits remain in the overall summary above. The group sizes and rank statistics are recomputed from the current run, and the exploratory p-values are descriptive.

In [ ]:
pattern_results = ar3_mae_results.copy()
pattern_results["minimum_width_wins"] = (
    pattern_results["coefficient_mae"]
    < pattern_results["chebyshev_coefficient_mae"]
)
pattern_results["geometric_failure"] = (
    pattern_results["coefficient_mae"] > 1.0
)
for coefficient_index in range(1, order + 1):
    pattern_results[f"abs_phi_{coefficient_index}"] = abs(
        pattern_results[f"true_phi_{coefficient_index}"]
    )
pattern_results["phi_norm"] = np.sqrt(
    sum(
        pattern_results[f"true_phi_{index}"] ** 2
        for index in range(1, order + 1)
    )
)
pattern_results["true_u_0"] = 1.0 / np.sqrt(
    1.0 + pattern_results["phi_norm"] ** 2
)
pattern_results["direction_error"] = (
    1.0 - pattern_results["direction_cosine_similarity"]
)
pattern_results["log10_relative_advantage"] = np.log10(
    pattern_results["chebyshev_coefficient_mae"]
    / pattern_results["coefficient_mae"]
)

from Code.Code_AR_processes.ar_tools.functions import stationarity_root_margin


pattern_results["root_margin"] = pattern_results.apply(
    stationarity_root_margin, axis=1, order=order
)
regular_results = pattern_results.loc[
    ~pattern_results["geometric_failure"]
].copy()
winning_results = regular_results.loc[
    regular_results["minimum_width_wins"]
]
losing_results = regular_results.loc[
    ~regular_results["minimum_width_wins"]
]

metric_columns = {
    "|phi_1|": "abs_phi_1",
    "|phi_2|": "abs_phi_2",
    "|phi_3|": "abs_phi_3",
    "||phi||": "phi_norm",
    "true |u_0|": "true_u_0",
    "direction error (1-cosine)": "direction_error",
    "stationarity root margin": "root_margin",
}
metric_rows = []
correlation_rows = []
for label, column in metric_columns.items():
    win_values = winning_results[column]
    loss_values = losing_results[column]
    if len(win_values) and len(loss_values):
        rank_test = mannwhitneyu(win_values, loss_values, alternative="two-sided")
        rank_biserial = 2.0 * rank_test.statistic / (len(win_values) * len(loss_values)) - 1.0
        rank_p_value = float(rank_test.pvalue)
    else:
        rank_biserial = np.nan
        rank_p_value = np.nan
    metric_rows.append({
        "metric": label,
        "win_median": float(win_values.median()),
        "loss_median": float(loss_values.median()),
        "rank_biserial_effect": float(rank_biserial),
        "exploratory_p_value": rank_p_value,
    })
    rank_correlation = spearmanr(
        regular_results[column],
        regular_results["log10_relative_advantage"],
    )
    correlation_rows.append({
        "metric": label,
        "spearman_rho_with_advantage": float(rank_correlation.statistic),
        "exploratory_p_value": float(rank_correlation.pvalue),
    })

win_pattern_summary = pd.DataFrame(metric_rows).set_index("metric")
advantage_correlations = pd.DataFrame(correlation_rows).set_index("metric")
relative_advantage = (
    pattern_results["chebyshev_coefficient_mae"]
    / pattern_results["coefficient_mae"]
)

print(f"Nominal minimum-width wins: {int(pattern_results['minimum_width_wins'].sum())}/50")         ### REPORTING
print(f"Wins by more than 10%: {int((relative_advantage > 1.1).sum())}/50")                         ### REPORTING
print(f"Wins by a factor above 2: {int((relative_advantage > 2.0).sum())}/50")                      ### REPORTING
print("\nMedian characteristics of regular wins and losses:")                                       ### REPORTING
print(win_pattern_summary.to_string())                                                              ### REPORTING
print("\nRank correlations with log10(LP MAE / minimum-width MAE):")                                ### REPORTING
print(advantage_correlations.to_string())                                                           ### REPORTING


## Four-method coefficient comparison for known AR order

This section is self-contained: run the setup cell below, then run the experiment cell. It draws 100 independent causal AR($p$) coefficient vectors for each $p=1,\ldots,5$ using `jones_ar_coefficients`. Each process has 2,000 retained observations after a 1,000-step burn-in and independent $\operatorname{Unif}[-1,1]$ innovations. All four methods use the same realization and the known order $p$ in each replication.

OLS minimizes squared residuals without an intercept; Yule-Walker solves the empirical autocovariance equations after subtracting the sample mean; Horváth's $L_\infty$ linear program minimizes the largest absolute residual without an intercept, as written in Section~\ref{sec:classical_estimation_methods}. Minimum width uses the $k=p+1$ embedding and the same 12-start, 5,000-iteration geometric descent settings as the AR(3) experiment above. For each order and method, the reported score is

$$\operatorname{MAE}_p=\frac{1}{100p}\sum_{l=1}^{100}\sum_{j=1}^{p}\left|\widehat\phi_j^{(l)}-\phi_j^{(l)}\right|.$$

Every replication contributes to the score, including large estimation errors. The experiment cell saves per-process estimates and the summary table under `outputs/csv/ar3_minimum_width_k4/four_method_comparison/absolute_steps/mae/` when you run it.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Resolve imports independently of the earlier, computationally expensive cells.
AR_CODE_DIR = next(
    (
        candidate.resolve()
        for root in (Path.cwd(), *Path.cwd().parents)
        for candidate in (root / "Code" / "Code_AR_processes", root)
        if (candidate / "ar_tools/functions.py").is_file()
    ),
    None,
)
if AR_CODE_DIR is None:
    raise FileNotFoundError("Could not locate Code/Code_AR_processes.")
PROJECT_ROOT = AR_CODE_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Code.Code_AR_processes.ar_tools.descent import batched_geometric_width_curves
from Code.Code_AR_processes.ar_tools.functions import (
    jones_ar_coefficients,
    simulate_uniform_ar,
    sliding_window_point_cloud,
)


from Code.Code_AR_processes.ar_tools.classical_selectors import fit_ols_known_order


from Code.Code_AR_processes.ar_tools.classical_selectors import fit_yule_walker_known_order


from Code.Code_AR_processes.ar_tools.classical_selectors import fit_horvath_known_order


from Code.Code_AR_processes.ar_tools.descent import fit_minimum_width_known_order

from Code.Code_AR_processes.ar_tools.coefficient_errors import coefficient_mae

In [ ]:
comparison_config = {
    "orders": [1, 2, 3, 4, 5],
    "coefficient_error_metric": "mae_per_coefficient_v1",
    "replications_per_order": 100,
    "sample_size": 2_000,
    "burnin": 1_000,
    "innovation_half_width": 1.0,
    "seed": 20260923,
    "minimum_width_starts": 12,
    "minimum_width_iterations": 5_000,
    "minimum_width_initial_step": 0.2,
    "minimum_width_geometric_decay": 0.997,
}
method_names = ["OLS (L2)", "Yule-Walker", "Horváth (L∞)", "Minimum width"]
comparison_rows = []

for true_order in comparison_config["orders"]:
    for replication in range(comparison_config["replications_per_order"]):
        coefficient_rng = np.random.default_rng(np.random.SeedSequence([
            comparison_config["seed"], true_order, replication, 0,
        ]))
        innovation_rng = np.random.default_rng(np.random.SeedSequence([
            comparison_config["seed"], true_order, replication, 1,
        ]))
        true_phi = jones_ar_coefficients(true_order, rng=coefficient_rng)
        series = simulate_uniform_ar(
            true_phi,
            sample_size=comparison_config["sample_size"],
            burnin=comparison_config["burnin"],
            half_width=comparison_config["innovation_half_width"],
            rng=innovation_rng,
        )
        point_cloud = sliding_window_point_cloud(series, true_order + 1)
        estimates = {
            "OLS (L2)": fit_ols_known_order(point_cloud),
            "Yule-Walker": fit_yule_walker_known_order(series, true_order),
            "Horváth (L∞)": fit_horvath_known_order(point_cloud),
            "Minimum width": fit_minimum_width_known_order(
                series,
                point_cloud,
                true_order,
                seed=comparison_config["seed"],
                replication=100 * true_order + replication,
                starts=comparison_config["minimum_width_starts"],
                iterations=comparison_config["minimum_width_iterations"],
                initial_step=comparison_config["minimum_width_initial_step"],
                decay=comparison_config["minimum_width_geometric_decay"],
            ),
        }
        for method in method_names:
            estimated_phi = np.asarray(estimates[method], dtype=float)
            if estimated_phi.shape != true_phi.shape or not np.all(np.isfinite(estimated_phi)):
                raise RuntimeError(
                    f"Invalid {method} estimate for AR({true_order}), "
                    f"replication {replication + 1}."
                )
            absolute_errors = np.abs(estimated_phi - true_phi)
            row = {
                "order": true_order,
                "replication": replication + 1,
                "method": method,
                "coefficient_mae": coefficient_mae(estimated_phi, true_phi),
            }
            for coefficient_index in range(true_order):
                suffix = coefficient_index + 1
                row[f"true_phi_{suffix}"] = true_phi[coefficient_index]
                row[f"estimated_phi_{suffix}"] = estimated_phi[coefficient_index]
                row[f"absolute_error_phi_{suffix}"] = absolute_errors[coefficient_index]
            comparison_rows.append(row)

        if (replication + 1) % 10 == 0:
            print(f"AR({true_order}): {replication + 1}/100 processes complete")

coefficient_comparison_results = pd.DataFrame(comparison_rows)
coefficient_comparison_summary = (
    coefficient_comparison_results
    .groupby(["order", "method"], as_index=False)
    .agg(MAE=("coefficient_mae", "mean"), process_count=("coefficient_mae", "size"))
)
assert (coefficient_comparison_summary["process_count"] == 100).all()
coefficient_comparison_table = (
    coefficient_comparison_summary
    .pivot(index="order", columns="method", values="MAE")
    .reindex(index=comparison_config["orders"], columns=method_names)
)

comparison_output_dir = (
    AR_CODE_DIR / "outputs" / "csv" / "ar3_minimum_width_k4" / "four_method_comparison" / "absolute_steps" / "mae"
)
comparison_output_dir.mkdir(parents=True, exist_ok=True)
coefficient_comparison_results.to_csv(
    comparison_output_dir / "per_process_estimates.csv", index=False
)
coefficient_comparison_summary.to_csv(
    comparison_output_dir / "mae_summary.csv", index=False
)
with (comparison_output_dir / "configuration.json").open("w") as config_file:
    json.dump(comparison_config, config_file, indent=2)

print("MAE = mean absolute error per coefficient, averaged over 100 processes")
print(coefficient_comparison_table.to_string(float_format=lambda value: f"{value:.6f}"))
print(f"Saved results to {comparison_output_dir}")

## Mean and median coefficient error versus AR order

The two panels compare all four known-order estimators for $p=1,\ldots,5$. Each point summarizes the same 100 simulated processes for its order. The vertical axes are logarithmic so that both the typical errors and the large, unfiltered minimum-width means remain visible. The plotting cell reads the saved experiment CSV files directly; it does not rerun the simulation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Read the completed experiment; no simulation or fitting is run here.
comparison_data_dir = next(
    (
        candidate.resolve()
        for root in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            root / "Code" / "Code_AR_processes" / "outputs" / "csv"
            / "ar3_minimum_width_k4" / "four_method_comparison" / "absolute_steps" / "mae",
            root / "outputs" / "csv" / "ar3_minimum_width_k4" / "mae"
            / "four_method_comparison" / "absolute_steps" / "mae",
        )
        if (candidate / "mae_summary.csv").is_file()
        and (candidate / "per_process_estimates.csv").is_file()
    ),
    None,
)
if comparison_data_dir is None:
    raise FileNotFoundError("Run the four-method experiment before plotting.")

mean_results = pd.read_csv(comparison_data_dir / "mae_summary.csv")
per_process_results = pd.read_csv(comparison_data_dir / "per_process_estimates.csv")
median_results = (
    per_process_results.groupby(["order", "method"], as_index=False)
    .agg(median_error=("coefficient_mae", "median"), process_count=("coefficient_mae", "size"))
)
plot_results = mean_results.merge(
    median_results,
    on=["order", "method", "process_count"],
    validate="one_to_one",
)
methods = ["OLS (L2)", "Yule-Walker", "Horváth (L∞)", "Minimum width"]
orders = [1, 2, 3, 4, 5]
assert len(plot_results) == len(methods) * len(orders)
assert (plot_results["process_count"] == 100).all()
assert (plot_results[["MAE", "median_error"]].to_numpy() > 0).all()

styles = {
    "OLS (L2)": ("#0072B2", "o"),
    "Yule-Walker": ("#E69F00", "s"),
    "Horváth (L∞)": ("#009E73", "D"),
    "Minimum width": ("#D55E00", "^"),
}
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.8), sharex=True)
for axis, column, title in zip(
    axes,
    ["MAE", "median_error"],
    ["Mean coefficient MAE", "Median coefficient MAE"],
):
    for method in methods:
        method_data = (
            plot_results.loc[plot_results["method"] == method]
            .set_index("order").loc[orders]
        )
        color, marker = styles[method]
        axis.plot(
            orders,
            method_data[column],
            label=method,
            color=color,
            marker=marker,
            linewidth=2.0,
            markersize=6,
        )
    axis.set(title=title, xlabel="True AR order $p$", xticks=orders)
    axis.set_yscale("log")
    axis.grid(which="major", color="0.85", linewidth=0.8)
    axis.set_axisbelow(True)
axes[0].set_ylabel("Coefficient MAE (log scale)")
axes[1].set_ylabel("Coefficient MAE (log scale)")
fig.suptitle("Known-order AR coefficient estimation", fontsize=14, y=0.98)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.015),
           ncol=4, frameon=False)
fig.subplots_adjust(left=0.09, right=0.98, top=0.83, bottom=0.23, wspace=0.28)

figure_name = "mean_and_median_mae_vs_order"
figure_base = next(parent for parent in comparison_data_dir.parents if parent.name == "outputs")
png_dir = figure_base / "images" / "ar3_minimum_width_k4" / "mae"
pdf_dir = figure_base / "pdfs" / "ar3_minimum_width_k4" / "mae"
png_dir.mkdir(parents=True, exist_ok=True)
pdf_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(png_dir / f"{figure_name}.png", dpi=220, bbox_inches="tight")
fig.savefig(pdf_dir / f"{figure_name}.pdf", bbox_inches="tight")
plt.show()

## How often is each estimator best?

For every simulated process, compare the four methods using coefficient MAE. The bars count the method with the smallest MAE among the paired estimates at each true order. The cell checks completeness and ties before counting wins.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Read the completed comparison; this cell does not simulate or fit processes.
comparison_data_dir = next(
    (
        candidate.resolve()
        for root in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            root / "Code" / "Code_AR_processes" / "outputs" / "csv"
            / "ar3_minimum_width_k4" / "four_method_comparison" / "absolute_steps" / "mae",
            root / "outputs" / "csv" / "ar3_minimum_width_k4" / "mae"
            / "four_method_comparison" / "absolute_steps" / "mae",
        )
        if (candidate / "per_process_estimates.csv").is_file()
    ),
    None,
)
if comparison_data_dir is None:
    raise FileNotFoundError("Run the four-method experiment before plotting.")

per_process_results = pd.read_csv(comparison_data_dir / "per_process_estimates.csv")
methods = ["OLS (L2)", "Yule-Walker", "Horváth (L∞)", "Minimum width"]
orders = [1, 2, 3, 4, 5]
errors_by_process = per_process_results.pivot(
    index=["order", "replication"], columns="method", values="coefficient_mae"
).reindex(columns=methods)
if (
    errors_by_process.shape != (500, 4)
    or errors_by_process.isna().any().any()
    or not errors_by_process.groupby(level="order").size().eq(100).all()
):
    raise ValueError("Expected four errors for each of 100 processes at each order.")

best_error = errors_by_process.min(axis=1)
tie_counts = errors_by_process.eq(best_error, axis=0).sum(axis=1)
if not tie_counts.eq(1).all():
    raise ValueError("A process has tied best methods; decide a tie rule first.")
best_method = errors_by_process.idxmin(axis=1).rename("best_method")
winner_counts = (
    best_method.reset_index()
    .groupby(["order", "best_method"]).size()
    .unstack(fill_value=0)
    .reindex(index=orders, columns=methods, fill_value=0)
)
assert winner_counts.sum(axis=1).eq(100).all()
winner_counts.rename_axis("order").to_csv(comparison_data_dir / "winner_counts.csv")

colors = {
    "OLS (L2)": "#0072B2",
    "Yule-Walker": "#E69F00",
    "Horváth (L∞)": "#009E73",
    "Minimum width": "#D55E00",
}
fig, axis = plt.subplots(figsize=(10.4, 5.3))
positions = np.arange(len(orders), dtype=float)
bar_width = 0.19
for method_index, method in enumerate(methods):
    counts = winner_counts[method].to_numpy()
    bars = axis.bar(
        positions + (method_index - 1.5) * bar_width,
        counts,
        width=bar_width,
        color=colors[method],
        label={"OLS (L2)": r"OLS ($L_2$)", "Horváth (L∞)": r"Horváth ($L_\infty$)"}.get(method, method),
    )
    axis.bar_label(bars, labels=[str(count) for count in counts],
                   padding=3, fontsize=8)

axis.set(
    title="Lowest coefficient MAE on each simulated process",
    xlabel="True AR order $p$",
    ylabel="Wins out of 100 processes",
    xticks=positions,
    xticklabels=[str(order) for order in orders],
    ylim=(0, 105),
)
axis.set_yticks(np.arange(0, 101, 20))
axis.grid(axis="y", color="0.85", linewidth=0.8)
axis.set_axisbelow(True)
axis.spines[["top", "right"]].set_visible(False)
axis.legend(loc="upper left", ncol=2, frameon=False)
fig.tight_layout()

figure_name = "best_method_frequency_by_order"
outputs_dir = next(parent for parent in comparison_data_dir.parents if parent.name == "outputs")
png_dir = outputs_dir / "images" / "ar3_minimum_width_k4" / "mae"
pdf_dir = outputs_dir / "pdfs" / "ar3_minimum_width_k4" / "mae"
png_dir.mkdir(parents=True, exist_ok=True)
pdf_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(png_dir / f"{figure_name}.png", dpi=220, bbox_inches="tight")
fig.savefig(pdf_dir / f"{figure_name}.pdf", bbox_inches="tight")
plt.show()

## First direction component versus coefficient MAE

The scatter plot uses the 50-run AR(3) experiment above. The left panel shows every fit; the right panel shows fits with coefficient MAE at most one. Both MAE axes are logarithmic. Spearman's rank correlation is calculated separately for each panel. Run the coefficient-recovery cells first to create the MAE data, or load the MAE CSV generated by a previous run of this cell.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import spearmanr

ar_code_dir = next(
    (
        candidate.resolve()
        for root in (Path.cwd(), *Path.cwd().parents)
        for candidate in (root / "Code" / "Code_AR_processes", root)
        if (candidate / "ar_tools/functions.py").is_file()
    ),
    None,
)
if ar_code_dir is None:
    raise FileNotFoundError("Could not locate Code/Code_AR_processes.")

scatter_csv = (
    ar_code_dir / "outputs" / "csv" / "ar3_minimum_width_k4" / "mae"
    / "mae" / "ar3_50_run_displayed_u0_mae.csv"
)
if "ar3_mae_results" in globals():
    scatter_data = ar3_mae_results[
        ["replication", "abs_u_0", "coefficient_mae"]
    ].copy()
    scatter_csv.parent.mkdir(parents=True, exist_ok=True)
    scatter_data.to_csv(scatter_csv, index=False)
else:
    scatter_data = pd.read_csv(scatter_csv)

if (
    len(scatter_data) != 50
    or scatter_data["replication"].nunique() != 50
    or not scatter_data[["abs_u_0", "coefficient_mae"]].notna().all().all()
    or not (scatter_data[["abs_u_0", "coefficient_mae"]] > 0).all().all()
):
    raise ValueError("Expected 50 complete, positive AR(3) observations.")

regular = scatter_data[scatter_data["coefficient_mae"] <= 1]
extreme = scatter_data[scatter_data["coefficient_mae"] > 1]
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.8), sharex=True)
for axis, data, title in (
    (axes[0], scatter_data, "All 50 runs"),
    (axes[1], regular, fr"{len(regular)} runs with MAE $\leq 1$"),
):
    base = data[data["coefficient_mae"] <= 1]
    axis.scatter(
        base["abs_u_0"], base["coefficient_mae"],
        s=48, color="#0072B2", alpha=0.82,
        edgecolors="white", linewidths=0.5, label=r"MAE $\leq 1$",
    )
    if axis is axes[0] and len(extreme):
        axis.scatter(
            extreme["abs_u_0"], extreme["coefficient_mae"],
            s=75, color="#D55E00", marker="^",
            edgecolors="white", linewidths=0.5, label="MAE $>1$",
        )
        for row in extreme.itertuples():
            axis.annotate(
                str(int(row.replication)),
                (row.abs_u_0, row.coefficient_mae),
                xytext=(5, 3), textcoords="offset points", fontsize=9,
            )
    rho, _ = spearmanr(data["abs_u_0"], data["coefficient_mae"])
    axis.text(
        0.97, 0.95, fr"Spearman $\rho={rho:.3f}$",
        transform=axis.transAxes, ha="right", va="top",
        bbox={"facecolor": "white", "edgecolor": "0.85", "alpha": 0.9},
    )
    axis.set(
        title=title,
        xlabel=r"$|\widehat{u}_0|$",
        ylabel="Coefficient MAE (log scale)",
        xlim=(-0.03, 1.02),
    )
    axis.set_yscale("log")
    axis.grid(which="major", color="0.85", linewidth=0.8)
    axis.set_axisbelow(True)
    axis.spines[["top", "right"]].set_visible(False)
axes[0].legend(loc="lower left", frameon=False)
fig.tight_layout()

figure_name = "abs_u0_vs_coefficient_mae_ar3_50_runs"
png_dir = ar_code_dir / "outputs" / "images" / "ar3_minimum_width_k4" / "mae"
pdf_dir = ar_code_dir / "outputs" / "pdfs" / "ar3_minimum_width_k4" / "mae"
png_dir.mkdir(parents=True, exist_ok=True)
pdf_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(png_dir / f"{figure_name}.png", dpi=220, bbox_inches="tight")
fig.savefig(pdf_dir / f"{figure_name}.pdf", bbox_inches="tight")
plt.show()

## Minimum-width coefficient MAE versus sample size

For each of 100 independently drawn causal AR(3) coefficient vectors, generate one length-2,000 path after burn-in with uniform innovations. Fit the minimum-width estimator at $k=4$ to the nested prefixes $T\in\{100,500,1000,1500,2000\}$ of that same path. This pairs the sample sizes within each process. At each $T$, report

$$\operatorname{MAE}(T)=\frac{1}{100}\sum_{l=1}^{100}\frac{1}{3}\sum_{j=1}^{3}\left|\widehat\phi_{j,T}^{(l)}-\phi_j^{(l)}\right|.$$

The optimizer uses the same 12-start geometric schedule as the 50-process experiment above (5,000 iterations, $\eta_0=0.2$, decay $0.997$). It evaluates widths on convex-hull vertices to speed up the run. Every finite error, including extreme errors, contributes to the mean. A failed fit is recorded and makes the unconditional MAE unavailable. Individual fits and the summary are saved under `outputs/csv/ar3_minimum_width_k4/mae_vs_T_100_processes/absolute_steps/mae/`. Run the next cell to execute or resume this experiment through the shared `ar_tools.experiments` helper.

In [ ]:
from pathlib import Path
import pandas as pd

from Code.Code_AR_processes.ar_tools.experiment_settings import AR3_MAE_OUTPUT_DIRECTORY as AR3_MAE_VS_T_OUTPUT_DIR
from Code.Code_AR_processes.ar_tools.experiments import run_ar3_coefficient_mae_experiment as run_ar3_mae_vs_T

run_ar3_mae_vs_T()  # Completed processes are loaded from the cache.
ar3_mae_vs_T_summary = pd.read_csv(AR3_MAE_VS_T_OUTPUT_DIR / "mae_summary.csv")
ar3_mae_vs_T_results = pd.read_csv(AR3_MAE_VS_T_OUTPUT_DIR / "replication_results.csv")
assert len(ar3_mae_vs_T_results) == 500
assert ar3_mae_vs_T_results.groupby("T").size().eq(100).all()
display(ar3_mae_vs_T_summary.round(8))
